# D01 — Replogle 2022 essential-gene Perturb-seq atlases

Downloads the K562 and RPE1 essential-gene Perturb-seq atlases (Replogle et al. 2022, *Cell*) via the `pertpy` interface to the scverse-harmonised mirror. These are the underlying single-cell datasets that CPA and GEARS are trained on in P01 and P02.

**Manuscript reference:** Methods §2.1 (Datasets and severity reference).

**Outputs:**
- `data/replogle/k562_essential.h5ad` (~5 GB)
- `data/replogle/rpe1_essential.h5ad` (~6 GB)

The downloads are cached by `pertpy` to `~/.cache/pertpy/` on first run; this notebook copies the cached files into the project's `data/replogle/` directory so the rest of the pipeline can find them at a deterministic path.

**Dependencies:** this notebook requires the `data` dependency group:
```bash
uv sync --group data
```


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR

OUT_DIR = DATA_DIR / "replogle"
OUT_DIR.mkdir(exist_ok=True, parents=True)


## Check whether pertpy is installed

If `pertpy` is not in the active environment, the notebook exits with instructions. The figure / analysis layers do not depend on this notebook.

In [ ]:
try:
    import pertpy as pt
    print(f"pertpy version: {pt.__version__}")
    pertpy_available = True
except ImportError:
    pertpy_available = False
    print("pertpy is not installed in the active environment.")
    print("Install with:  uv sync --group data")
    print()
    print("This notebook downloads the Replogle 2022 K562 and RPE1 atlases used")
    print("by P01 (CPA training) and P02 (GEARS training). If you only need to")
    print("reproduce the figures and tables, run:")
    print("  ./run_all.sh --figures")
    print("which uses the precomputed evaluation CSVs shipped in precomputed/eval/.")


## Download the atlases

In [ ]:
if pertpy_available:
    import shutil

    targets = [
        ("K562", "k562_essential.h5ad", "replogle_2022_k562_essential"),
        ("RPE1", "rpe1_essential.h5ad", "replogle_2022_rpe1"),
    ]

    for cell_type, fname, method in targets:
        out_path = OUT_DIR / fname
        if out_path.exists():
            print(f"{cell_type}: {out_path} already exists ({out_path.stat().st_size / 1e9:.2f} GB), skipping download")
            continue

        print(f"{cell_type}: downloading via pertpy.data.{method}() ...")
        adata = getattr(pt.data, method)()
        adata.write_h5ad(out_path)
        print(f"  wrote {out_path} ({out_path.stat().st_size / 1e9:.2f} GB)")
        print(f"  shape: {adata.shape}  (cells x genes)")
        print(f"  perturbations: {adata.obs['perturbation'].nunique() if 'perturbation' in adata.obs else 'unknown'}")

    print("\nAll downloads complete.")
    for cell_type, fname, _ in targets:
        out_path = OUT_DIR / fname
        if out_path.exists():
            print(f"  {cell_type}: {out_path}  ({out_path.stat().st_size / 1e9:.2f} GB)")
